In [60]:
# Preamble. First block is for adding packages in the notebook environment. Second block declares use and establishes a 
# namespace for the specified packages.
using Pkg
Pkg.add("CSV")
Pkg.add("DataFrames")
Pkg.add("GLM")
Pkg.add("Statistics")

using CSV, DataFrames, GLM, Statistics

   Resolving package versions...
  No Changes to `C:\Users\hapra\.julia\environments\v1.9\Project.toml`
  No Changes to `C:\Users\hapra\.julia\environments\v1.9\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\hapra\.julia\environments\v1.9\Project.toml`
  No Changes to `C:\Users\hapra\.julia\environments\v1.9\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\hapra\.julia\environments\v1.9\Project.toml`
  No Changes to `C:\Users\hapra\.julia\environments\v1.9\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\hapra\.julia\environments\v1.9\Project.toml`
  No Changes to `C:\Users\hapra\.julia\environments\v1.9\Manifest.toml`


# File Import

## Python Source
~~~
M0StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FundMStarData.csv',delimiter=',',skip_header=0))) # No header, so don't skip
M1StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FirstOvertoneMStarData.csv',delimiter=',',skip_header=0))) # Has header, so skip
C0StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FundamentalCStarData.csv',delimiter=',',skip_header=0))) # Has header, so skip
C1StarData = np.transpose(np.asarray(np.genfromtxt('ObservablePlus_FirstOvertoneCStarData.csv',delimiter=',',skip_header=0))) # No header, so don't skip

print(np.shape(M0StarData))
print(np.shape(M1StarData))
print(np.shape(C0StarData))
print(np.shape(C1StarData))




MdotIndex=5 # Column of imported files that has Mdot data
MdotErrorIndex=6 # Absolute error in Mdot
MdotDustIndex=2 # Column of imported files that has Mdot_dust Data
LIndex=8 # Column of imported files that has LBol data
LErrorIndex=9 # Absolute error in L
PIndex=13 # Column of imported files that has Period data
JIndex=14 # Column of imported files that has J-band magnitude
HIndex=15 # Column of imported files that has J-band magnitude
KIndex=16 # Column of imported files that has J-band magnitude
ThreePointSixIndex=17 # Column of imported files that has 3.6 band magnitude
FourPointFiveIndex=18 # Column of imported files that has 3.6 band magnitude
FivePointEightIndex=19 # Column of imported files that has 3.6 band magnitude
EightPointZeroIndex=20 # Column of imported files that has 8.0 band magnitude
MIndex=22 # Column of imported files that has Mass data
RIndex=23 # Column of imported files that has Radius data
TeffObsvIndex=12 # Column of imported files that has Observed Teff Data
TeffCalcIndex=24 # Column of imported files that has Calculated Teff Data
tauIndex=27 # Column of imported files that has Optical Depth data
AmpIndex=28 # Column of imported files that has pulsation amplitude data
~~~

In [16]:
# Import our data on AGB stars into Julia arrays.
M0StarData = CSV.read("ObservablePlus_FundMStarData.csv", DataFrame, header = false)
M1StarData = CSV.read("ObservablePlus_FundCStarData.csv", DataFrame, header = false)
C0StarData = CSV.read("ObservablePlus_FirstOvertoneMStarData.csv", DataFrame, header = false)
C1StarData = CSV.read("ObservablePlus_FirstOvertoneCStarData.csv", DataFrame, header = false)
print("Data ''ObservablePlus_FundMStarData'', ''ObservablePlus_FundCStarData'', ''ObservablePlus_FirstOvertoneMStarData'', and ''ObservablePlus_FirstOvertoneCStarData'' successfully imported to arrays.")

Data ''ObservablePlus_FundMStarData'', ''ObservablePlus_FundCStarData'', ''ObservablePlus_FirstOvertoneMStarData'', and ''ObservablePlus_FirstOvertoneCStarData'' successfully imported to arrays.

In julia, arrays are called with the format array[foo,bar]. Indices can be replaced by the protected character ":", which calls all rows or columns of the array.

In [25]:
#= The following block assigns indices to each of the variables in the '*StarData.csv' files of known format. Note, julia is
indexed from 1, so these are shifted up by 1 compared to the previous Python3 version of this algorithm. If a catalog with a 
different format is desired, simply change these values to match the new catalog.
=#
MdotIndex=6 # Column of imported files that has Mdot data
MdotErrorIndex=7 # Absolute error in Mdot
MdotDustIndex=3 # Column of imported files that has Mdot_dust Data
LIndex=9 # Column of imported files that has LBol data
LErrorIndex=10 # Absolute error in L
PIndex=14 # Column of imported files that has Period data
JIndex=15 # Column of imported files that has J-band magnitude
HIndex=16 # Column of imported files that has H-band magnitude
KIndex=17 # Column of imported files that has K-band magnitude
ThreePointSixIndex=18 # Column of imported files that has 3.6 micron band magnitude
FourPointFiveIndex=19 # Column of imported files that has 4.5 micron band magnitude
FivePointEightIndex=20 # Column of imported files that has 5.8 micron band magnitude
EightPointZeroIndex=21 # Column of imported files that has 8.0 micron band magnitude
MIndex=23 # Column of imported files that has Mass data
RIndex=24 # Column of imported files that has Radius data
TeffObsvIndex=13 # Column of imported files that has Observed Teff Data
TeffCalcIndex=25 # Column of imported files that has Calculated Teff Data
tauIndex=28 # Column of imported files that has Optical Depth data
AmpIndex=29 # Column of imported files that has pulsation amplitude data
print("Indices assigned for data retrevial.")

Indices assigned for data retrevial.

# Fitting Methods

## Python Source

~~~
def fit_seek_one_percentile_m (star_data, initial_n=0.5, terminal_n=3.0, delta_n=0.1,percentile_to_trim = 0.05, 
              spec_mode_label ='', digit_precision = 6,text_x=0.03):
    '''
    Goal: script that calculates A, B, C for fit. Then, take those values and compare them to ideal. Knowing that,
    compare to previous fit. If better, continue. If worse, increase resolution then continue from previous.
    Repeat until a limit on the number of iterations is reached or we are as close to 1 as we can measure with
    numerical precision.
    '''
    '''
    First, we will handle all of the calculations that are simply properties of the data set. No need to recalculate
    these on every iteration. These calculations are finding the line of best fit for the data (presumably the deathline),
    finding the standard deviation of the data around the deathline, calculating the vertical displacement of the upper and 
    lower bounds given a multiplier n, calculating the horizontal displacement of the upper and lower bounds from the fit line,
    using those two calculations to determine the height and width of the strip (respectively), and sorting out the data found 
    inside our LM strip.
    '''
    logLData = np.log10(star_data[LIndex])
    logMData = np.log10(star_data[MIndex])
    logMdotData = np.log10(star_data[MdotIndex])     
    # Take data, fit logM v logL (this line is the deathline)
    deathline_fit = linear_fit_1var(logLData,logMData)
    deathline_coeffs = deathline_fit[0]
    print('deathline_fit:',deathline_fit)
    # Plot logM v logL and fit line for checking
    line_x_vals = np.asarray([np.nanmin(np.log10(star_data[LIndex])),np.nanmax(np.log10(star_data[LIndex]))])
    line_y_vals = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals
    #bound_size = std_multiplier*np.nanstd(np.log10(star_data[MIndex]))
    stnd_dev_around_DL = np.nanstd(logMData-deathline_coeffs[0]-deathline_coeffs[1]*logLData)
                                   
    i = 0
    n = initial_n
    #delta_n = 0.1 # An initial incrementing value. We'll start with rough adjustment and change this as we get closer to a good fit
    early_cancel_flag = False
    
    n_array = []
    Exponents_array = []
    fit_qual_array = []
    stand_dev_array = []
    rsq_array = []
    deltas_array = []
    stars_in_bound_array = []
    while n<=terminal_n:
        n_array.append(n)
        bound_size = n*stnd_dev_around_DL*np.cos(np.arctan(deathline_coeffs[1]))
        line_y_vals_up = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals + bound_size
        line_y_vals_down = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals - bound_size
        # Find width of deltaLogL and deltaLogM using fit line, and plot them
        logL_left = (np.average(line_y_vals) - deathline_coeffs[0]-bound_size)/deathline_coeffs[1]
        logL_right = (np.average(line_y_vals) - deathline_coeffs[0]+bound_size)/deathline_coeffs[1]
        delta_logL = logL_right - logL_left
        delta_logM = line_y_vals_up[1] - line_y_vals_down[1]
        star_data_inside_bound = np.asarray([])
        for l in range(0, len(star_data[MIndex])):
            if logMData[l] <= deathline_coeffs[0] + deathline_coeffs[1]*logLData[l] + bound_size:
                if logMData[l] >= deathline_coeffs[0] + deathline_coeffs[1]*logLData[l] - bound_size:
                    if len(star_data_inside_bound) == 0:
                        star_data_inside_bound = np.transpose(star_data)[l]
                    else:
                        star_data_inside_bound = np.vstack((star_data_inside_bound,np.transpose(star_data)[l]))
        star_data_inside_bound = np.transpose(star_data_inside_bound)
        logMdotBoundedData = np.log10(star_data_inside_bound[MdotIndex])
        logMdotBoundedData_mean = np.nanmean(logMdotBoundedData)
        logMdotBoundedData_std = np.nanstd(logMdotBoundedData)
        logMBoundedData = np.log10(star_data_inside_bound[MIndex])
        logLBoundedData = np.log10(star_data_inside_bound[LIndex])
        num_in_bound = len(logMdotBoundedData)
        stars_in_bound_array.append(num_in_bound)

        '''
        Now onto the loop. For some number of iterations, we are going to seek better fits for logMdot = A + BlogL + ClogM.
        From previous experimentation (see above function, fit_golf), we know that as we adjust m/n, the significance of the fit
        is more-or-less fixed using this method. Therefore, we can keep iterating to improve the quality of the fit. We are doing
        this over n and are adjusting m to be to trim the most extreme mass-loss rates
        '''
        logMdot_spread_from_mean = np.abs(logMdotBoundedData - logMdotBoundedData_mean)
        num_to_check = int(np.round_(percentile_to_trim*len(logMdot_spread_from_mean)))
        #print('num_to_check:',num_to_check)
        indices_to_trim = logMdot_spread_from_mean.argsort()[-num_to_check:]
        trimmed_logMdotBoundedData = np.delete(logMdotBoundedData,indices_to_trim)
        delta_logMdot = np.nanmax(trimmed_logMdotBoundedData)-np.nanmin(trimmed_logMdotBoundedData)
        deltas_array.append([delta_logL, delta_logM,delta_logMdot])
        #delta_logMdot = m*logMdotBoundedData_std
        # Calculate B = deltaLogMdot/deltaLogL and C = deltaLogMdot/deltaLogM
        B = delta_logMdot/delta_logL
        C = -delta_logMdot/delta_logM
        # Simpler, we can force A to be whatever is necessary for the mean residual to be 0
        # <logMdot_obsv - logMdot_calc> = 0 = sum(logMdot_obsv[i] - (A + BlogL[i] + ClogM[i]))/len(data)  
        # => <A> = sum(logMdot_obsv[i] - BlogL[i] - ClogM[i])/len(data)
        A = 0
        for k in range(0, np.shape(star_data)[1]):
            if np.isnan(logMdotData[k])==True:
                #print('Index ',i,' skipped due to NaN in logMdot')
                continue
            elif np.isnan(logLData[k])==True:
                #print('Index ',i,' skipped due to NaN in logL')
                continue
            elif np.isnan(logMData[k])==True:
                #print('Index ',i,' skipped due to NaN in logM')
                continue
            else:
                A = A + logMdotData[k]-B*logLData[k] - C*logMData[k]
        A = A/np.shape(star_data)[1]
        logMdot_predict = A + B*np.log10(star_data[LIndex]) + C*np.log10(star_data[MIndex])
        correction = np.nanmean(np.log10(star_data[MdotIndex])-logMdot_predict)
        A = A + correction
        calc_logMdot = A + B*logLData + C*logMData
        Exponents_array.append([A,B,C])
        fit = linear_fit_1var(logMdotData, calc_logMdot)
        #print('fit:',fit)
        fit_params = fit[0]
        fitRSq = fit[1]
        fit_qual_array.append(fit_params[1])
        rsq_array.append(fitRSq)
        residuals = logMdotData-calc_logMdot
        stand_dev_array.append( np.nanstd(residuals))
    
        n = n + delta_n
    fit_qual_array = np.asarray(fit_qual_array)
    fit_qual_testing = np.abs(fit_qual_array - 1)
    where_min = np.argmin(fit_qual_testing)
    
    print('Optimal fit found at ',where_min)
    print('n=',n_array[where_min])
    print('Fit quality: Slope=',fit_qual_array[where_min])
    print('Fit significance: R^2=', rsq_array[where_min])
    print('Fit standard dev: sigma=', stand_dev_array[where_min])
    print('delta logL, delta logM, delta logMdot:', deltas_array[where_min])
    print('log A, B, C:', Exponents_array[where_min])
    print('# Stars in Bound:', stars_in_bound_array[where_min])
    print('\n')
    
    print('Lower bound fit:')
    print('n=',n_array[0])
    print('Fit quality: Slope=',fit_qual_array[0])
    print('Fit significance: R^2=', rsq_array[0])
    print('Fit standard dev: sigma=', stand_dev_array[0])
    print('delta logL, delta logM, delta logMdot:', deltas_array[0])
    print('log A, B, C:', Exponents_array[0])
    print('# Stars in Bound:', stars_in_bound_array[0])
    print('\n')
    
    print('Upper bound fit:')
    print('n=',n_array[-1])
    print('Fit quality: Slope=',fit_qual_array[-1])
    print('Fit significance: R^2=', rsq_array[-1])
    print('Fit standard dev: sigma=', stand_dev_array[-1])
    print('delta logL, delta logM, delta logMdot:', deltas_array[-1])
    print('log A, B, C:', Exponents_array[-1])
    print('# Stars in Bound:', stars_in_bound_array[-1])
    print('\n')
    
    # Get all of the looping values with our optimal n
    n= n_array[where_min]
    bound_size = n*stnd_dev_around_DL*np.cos(np.arctan(deathline_coeffs[1]))
    line_y_vals_up = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals + bound_size
    line_y_vals_down = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals - bound_size
    # Find width of deltaLogL and deltaLogM using fit line, and plot them
    logL_left = (np.average(line_y_vals) - deathline_coeffs[0]-bound_size)/deathline_coeffs[1]
    logL_right = (np.average(line_y_vals) - deathline_coeffs[0]+bound_size)/deathline_coeffs[1]
    delta_logL = logL_right - logL_left
    delta_logM = line_y_vals_up[1] - line_y_vals_down[1]
    star_data_inside_bound = np.asarray([])
    for l in range(0, len(star_data[MIndex])):
        if logMData[l] <= deathline_coeffs[0] + deathline_coeffs[1]*logLData[l] + bound_size:
            if logMData[l] >= deathline_coeffs[0] + deathline_coeffs[1]*logLData[l] - bound_size:
                if len(star_data_inside_bound) == 0:
                    star_data_inside_bound = np.transpose(star_data)[l]
                else:
                    star_data_inside_bound = np.vstack((star_data_inside_bound,np.transpose(star_data)[l]))
    star_data_inside_bound = np.transpose(star_data_inside_bound)
    logMdotBoundedData = np.log10(star_data_inside_bound[MdotIndex])
    logMdotBoundedData_mean = np.nanmean(logMdotBoundedData)
    logMdotBoundedData_std = np.nanstd(logMdotBoundedData)
    logMBoundedData = np.log10(star_data_inside_bound[MIndex])
    logLBoundedData = np.log10(star_data_inside_bound[LIndex])
    logMdot_spread_from_mean = np.abs(logMdotBoundedData - logMdotBoundedData_mean)
    num_to_check = int(np.round_(percentile_to_trim*len(logMdot_spread_from_mean)))
    #print('num_to_check:',num_to_check)
    indices_to_trim = logMdot_spread_from_mean.argsort()[-num_to_check:]
    #print('indices_to_trim',indices_to_trim)
    trimmed_logMdotBoundedData = np.delete(logMdotBoundedData,indices_to_trim)
    #print('Indices removed?:')
    delta_logMdot = np.nanmax(trimmed_logMdotBoundedData)-np.nanmin(trimmed_logMdotBoundedData)
    deltas_array.append([delta_logL, delta_logM,delta_logMdot])
    #delta_logMdot = m*logMdotBoundedData_std
    # Calculate B = deltaLogMdot/deltaLogL and C = deltaLogMdot/deltaLogM
    B = delta_logMdot/delta_logL
    C = -delta_logMdot/delta_logM
    # Simpler, we can force A to be whatever is necessary for the mean residual to be 0
    # <logMdot_obsv - logMdot_calc> = 0 = sum(logMdot_obsv[i] - (A + BlogL[i] + ClogM[i]))/len(data)  
    # => <A> = sum(logMdot_obsv[i] - BlogL[i] - ClogM[i])/len(data)
    A = 0
    for k in range(0, np.shape(star_data)[1]):
        if np.isnan(logMdotData[k])==True:
            #print('Index ',i,' skipped due to NaN in logMdot')
            continue
        elif np.isnan(logLData[k])==True:
            #print('Index ',i,' skipped due to NaN in logL')
            continue
        elif np.isnan(logMData[k])==True:
            #print('Index ',i,' skipped due to NaN in logM')
            continue
        else:
            A = A + logMdotData[k]-B*logLData[k] - C*logMData[k]
    A = A/np.shape(star_data)[1]
    logMdot_predict = A + B*np.log10(star_data[LIndex]) + C*np.log10(star_data[MIndex])
    correction = np.nanmean(np.log10(star_data[MdotIndex])-logMdot_predict)
    A = A + correction
    calc_logMdot = A + B*logLData + C*logMData
    fit = linear_fit_1var(logMdotData, calc_logMdot)
    fit_params = fit[0]
    fitRSq = fit[1]
    residuals = logMdotData-calc_logMdot
    
    
    # Plot 1: The LM Strip
    fig, ax = plt.subplots()
    plot.scatter(np.log10(star_data[LIndex]), np.log10(star_data[MIndex]),s=2)
    plot.plot(line_x_vals,line_y_vals,linestyle='-',color='black')
    plot.plot(line_x_vals,line_y_vals_up,linestyle='--',color='black')
    plot.plot(line_x_vals,line_y_vals_down,linestyle='--',color='black')
    plot.xlabel(r'$\log L$')
    plot.ylabel(r'$\log M$')
    #plot.title(spec_mode_label)
    ax.text(text_x, .97,spec_mode_label,horizontalalignment='center',verticalalignment='center',transform = ax.transAxes,fontsize=10)
    plot.vlines(np.average(line_x_vals),np.average(line_y_vals)-delta_logM/2,np.average(line_y_vals)+delta_logM/2,color='tab:green',linestyle='-',linewidth=4)
    plot.plot([logL_left,logL_right],[np.average(line_y_vals),np.average(line_y_vals)],color='tab:purple',linewidth=4)
    if type(n) == float:
        plot_name = 'deathzone_'+spec_mode_label+'_stnd_dev_'+str(n)[:4]
    #elif std_multiplier/(1/2*std_multiplier) == 2:
    #    plot_name = 'deathzone_'+spec_mode_label+'_stnd_dev_of_all_double'
    else:
        plot_name = 'deathzone_'+spec_mode_label+'_stnd_dev_of_all'
    plot.savefig(plot_name+'.png')
    plot.savefig(plot_name+'.pdf') # Why do journals require this format...
    plot.show()
    plot.close('all')
    
    # Plot 2: logMdot Histograms
    fig, ax = plt.subplots()
    n_hist, bins_hist, patches_hist = plot.hist(np.log10(star_data_inside_bound[MdotIndex]))
    ax.text(text_x, .97,spec_mode_label,horizontalalignment='center',verticalalignment='center',transform = ax.transAxes,fontsize=10)
    print('Full range:',np.nanmax(logMdotBoundedData),np.nanmin(logMdotBoundedData))
    print('vlines locations:',np.nanmax(trimmed_logMdotBoundedData),np.nanmin(trimmed_logMdotBoundedData))
    plot.vlines([np.nanmax(trimmed_logMdotBoundedData),np.nanmin(trimmed_logMdotBoundedData)],0,np.nanmax(n_hist))
    plot.xlabel(r'$\log \dot{M}$')
    if type(n) == int:
        plot_name = 'logMdot_histogram_'+spec_mode_label+'_stnd_dev_'+str(n)
    elif type(n) == float:
        plot_name = 'logMdot_histogram_'+spec_mode_label+'_stnd_dev_'+str(n)[0:5]
    else:
        plot_name = 'logMdot_histogram_'+spec_mode_label+'_stnd_dev_of_all'
    #plot.legend(loc=4)
    plot.savefig(plot_name+'.png')
    plot.savefig(plot_name+'.pdf') # Why do journals require this format...
    
    # Plot 3: log(Mdot/M) Histograms
    fig, ax = plt.subplots()
    n_hist, bins_hist, patches_hist = plot.hist(np.log10(star_data_inside_bound[MdotIndex]/star_data_inside_bound[MIndex]))
    ax.text(text_x, .97,spec_mode_label,horizontalalignment='center',verticalalignment='center',transform = ax.transAxes,fontsize=10)
    #print('Full range:',np.nanmax(logMdotBoundedData),np.nanmin(logMdotBoundedData))
    #print('vlines locations:',np.nanmax(trimmed_logMdotBoundedData),np.nanmin(trimmed_logMdotBoundedData))
    #plot.vlines([np.nanmax(trimmed_logMdotBoundedData),np.nanmin(trimmed_logMdotBoundedData)],0,np.nanmax(n_hist))
    plot.xlabel(r'$\log (\dot{M}/M)$')
    if type(n) == int:
        plot_name = 'logMdot_per_M_histogram_'+spec_mode_label+'_stnd_dev_'+str(n)
    elif type(n) == float:
        plot_name = 'logMdot_per_M_histogram_'+spec_mode_label+'_stnd_dev_'+str(n)[0:5]
    else:
        plot_name = 'logMdot_per_M_histogram_'+spec_mode_label+'_stnd_dev_of_all'
    #plot.legend(loc=4)
    plot.savefig(plot_name+'.png')
    plot.savefig(plot_name+'.pdf') # Why do journals require this format...
    
    
    return A, B, C
~~~

In [72]:
testlogLData = log10.((M0StarData[:,LIndex])) # Calculate the common log of the luminosities of the input data set.
testlogMData = log10.((M0StarData[:,MIndex])) # Calculate the common log of the masses of the input data set.

ols = lm(@formula(Y ~ X), DataFrame(X=testlogLData, Y=testlogMData)) 
coefs = coef(ols)
print("Coef:",coefs)
stnd_dev = std(testlogMData .- coefs[1]- coefs[2].*testlogLData)

Coef:[-2.1011267298869325, 0.599239588142298]

0.06566601075100557

In [101]:
ols

StatsModels.TableRegressionModel{LinearModel{GLM.LmResp{Vector{Float64}}, GLM.DensePredChol{Float64, LinearAlgebra.CholeskyPivoted{Float64, Matrix{Float64}, Vector{Int64}}}}, Matrix{Float64}}

Y ~ 1 + X

Coefficients:
─────────────────────────────────────────────────────────────────────────
                Coef.  Std. Error       t  Pr(>|t|)  Lower 95%  Upper 95%
─────────────────────────────────────────────────────────────────────────
(Intercept)  -2.10113   0.0308942  -68.01    <1e-99  -2.16172   -2.04054
X             0.59924   0.0087366   68.59    <1e-99   0.582106   0.616373
─────────────────────────────────────────────────────────────────────────

In [155]:
M0StarData[1,:]

Row,Column1,Column2,Column3,Column4,Column5,Column6,Column7,Column8,Column9,Column10,Column11,Column12,Column13,Column14,Column15,Column16,Column17,Column18,Column19,Column20,Column21,Column22,Column23,Column24,Column25,Column26,Column27,Column28,Column29
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,29.0,0.0,2.0e-10,5.0e-11,0.25,2.82322e-7,7.05804e-8,0.25,3075.0,221.0,0.0718699,3149.14,3700.0,144.55,12.9217,12.0548,11.8404,11.5562,11.5033,11.3889,11.2302,0.325946,0.991147,146.095,3555.84,3577.08,3.77,0.0256,0.651


In [170]:
r = rand(0:100, 5) # Values

5-element Vector{Int64}:
 92
  5
 29
 96
 24

In [172]:
i = sortperm(r) # What is the rank of value

5-element Vector{Int64}:
 2
 5
 3
 1
 4

In [195]:
test_logLData = log10.(M0StarData[!,LIndex])
isnan(test_logLData[10])

false

In [287]:
function fit_seek_one_percentile_m(star_data , y_data_type, chemistry = "M", pulsation_mode = "fundamental", skip_Mdot = false, initial_n=0.5, terminal_n=3.0, delta_n=0.1,percentile_to_trim = 0.05) #, 
    #=
    Goal: script that calculates A, B, C for fit. Then, take those values and compare them to ideal. Knowing that,
    compare to previous fit. If better, continue. If worse, increase resolution then continue from previous.
    Repeat until a limit on the number of iterations is reached or we are as close to 1 as we can measure with
    numerical precision.
    
    First, we will handle all of the calculations that are simply properties of the data set. No need to recalculate
    these on every iteration. These calculations are finding the line of best fit for the data (presumably the deathline),
    finding the standard deviation of the data around the deathline, calculating the vertical displacement of the upper and 
    lower bounds given a multiplier n, calculating the horizontal displacement of the upper and lower bounds from the fit line,
    using those two calculations to determine the height and width of the strip (respectively), and sorting out the data found 
    inside our LM strip.
    
    This method has two versions, one that fits the strip in L-M space and another that fits the strip in L-P space. The first 
    version can be picked by setting y_data_type = "M". The second version can be picked by setting y_data_type = "P". In the 
    case that we don't have mass-loss data to calibrate, we can disable all calculations involving the mass-loss rate by setting
    skip_Mdot = true.
    
    percentile_trim is an option that let's us exclude the most extreme mass-losing stars from the analysis; by defaut we cut 
    out the top 5%.
    
    chemistry and pulsation_mode let us specify the dust chemistry and pulsation mode of the stars in the data set. This lets us
    use the correct PMR and RML relations for determining the exponents for the power law we didn't determine from fitting. By
    default, we assume oxygen-rich (M) chemistry and the pulsation in the fundamental mode. Options for chemistry are 
    "M"/"oxygen-rich" and "C"/"carbon-rich". Options for mode are "fundamental"/"0" and "first-overtone"/"1".
    =#
    A = 0 # Initialize the return values, in case something fails or we skip a step that would make this return meaningless
    B = 0
    C = 0
    if y_data_type == "M"
        yIndex = MIndex
    elseif y_data_type == "P"
        yIndex = PIndex
    else
        print("Error: Invalid Data Type")
    end
    logLData = log10.((star_data[:,LIndex])) # Calculate the common log of the luminosities of the input data set.
    logyData = log10.((star_data[:,yIndex])) # Calculate the common log of the masses or periods of the input data set.
    if skip_Mdot == false
        logMdotData = log10.((star_data[:,MdotIndex])) # Calculate the common log of the luminosities of the input data set.
    end
    # Take data, fit logM v logL (this line is the deathline)
    deathline_fit = lm(@formula(Y ~ X), DataFrame(X=logLData, Y=logyData)) # Fit the deathline of the log L and log M data
    #print("Debug: Checkpoint 1")
    deathline_coeffs = coef(deathline_fit) # Retrieve the coefficients of that fit
    print("deathline_fit:",deathline_fit,"\n") # Provide the user with a print-out of the fit.
    line_x_vals = [minimum(logLData), maximum(logLData)]
    line_y_vals = deathline_coeffs[1] .+ deathline_coeffs[2]*line_x_vals
    stnd_dev_around_DL = std(logyData .-deathline_coeffs[1] - deathline_coeffs[2].*logLData)
    #=
    Now onto the loop. For some number of iterations, we are going to seek better fits for logMdot = A + BlogL + ClogM.
    From previous experimentation, we know that as we adjust m/n, the significance of the fit is more-or-less fixed using 
    this method. Therefore, we can keep iterating to improve the quality of the fit. We are doing this over n and are 
    adjusting m to trim the most extreme mass-loss rates.
    =#
    i = 0
    n = initial_n
    n_array = []
    Exponents_array = []
    stderror_array = []
    fit_qual_array = []
    rsq_array = []
    deltas_array = []
    stars_in_bound_array = []
    while n <= terminal_n
        #= The first step of this loop is to calculate the positions of the bounding lines of strip. This bound size is defined 
        vertically from the death line and it take to be some fitting value ''n'' times the std. dev. of the data around the 
        deathline times the cosine of the arctangent of the slope of the deathline.
        =#
        append!(n_array,n) # Add the current value of n to the list of values we have tested
        bound_size = (n*stnd_dev_around_DL*cos(atan(deathline_coeffs[2]))) # Calculate the bound size
        line_y_vals_up = deathline_coeffs[1] .+ deathline_coeffs[2].*line_x_vals .+ bound_size # Calculate the y-values of the upper bound
        line_y_vals_down = deathline_coeffs[1] .+ deathline_coeffs[2].*line_x_vals .- bound_size # Calculate the y-values of the lower bound
        #= Find width of deltaLogL and deltaLogM using fit line, and plot them =# 
        logL_left = (mean(line_y_vals) - deathline_coeffs[1]-bound_size)/deathline_coeffs[2] # Calculate the luminosity values for the upper bound
        logL_right = (mean(line_y_vals) - deathline_coeffs[1]+bound_size)/deathline_coeffs[2] # Calculate the luminosity values for the lower bound
        delta_logL = logL_right - logL_left # Calculate the width of the strip
        delta_logy = line_y_vals_up[2] - line_y_vals_down[2] # Calculate the height of the strip 
        #=print("Δlog(L)=",delta_logL,"\n")
        if y_data_type == "M"
            print("Δlog(M)=",delta_logy,"\n")
        elseif y_data_type == "P"
            print("Δlog(P)=",delta_logy,"\n")
        end
        =#
        #= Next, we will loop over the list of stars and determine which entries are within the strip=#
        indices_in_bounds = [] # Empty array
        for l in 1:nrow(star_data) # for all of the entries in star_data
            if (logyData[l] <= (deathline_coeffs[1] + deathline_coeffs[2]*logLData[l] + bound_size)) & (logyData[l] >= (deathline_coeffs[1] + deathline_coeffs[2]*logLData[l]-bound_size)) # if we are between the established bounds
                append!(indices_in_bounds,l) # Save l as an row that is in the bounds
            end
        end
        star_data_inside_bound = star_data[indices_in_bounds,:] # Keep only the indices that are inside bounds
        #print("star_data_inside_bound:",star_data_inside_bound,"\n")
        logMBoundedData = log10.(star_data_inside_bound[!,MIndex]) # Get the log of the mass of all stars in the bound
        logPBoundedData = log10.(star_data_inside_bound[!,PIndex]) # Get the log of the pulsation period of all stars in the bound
        logLBoundedData = log10.(star_data_inside_bound[!,LIndex]) # Get the log of the luminosity of all stars in the bound
        num_in_bound = nrow(star_data_inside_bound) # Calculate how many stars are included in the bound
        #print("num_in_bound=",num_in_bound,"\n")
        append!(stars_in_bound_array, num_in_bound) # Save the number of stars in this bound to an array "stars_in_bound_array"
        if skip_Mdot == false
            logMdotBoundedData = log10.(star_data_inside_bound[!,MdotIndex]) # Get the log of the mass-loss rate of all stars within the bound
            logMdotBoundedData_mean = mean(logMdotBoundedData) # Get mean of the above set
            logMdotBoundedData_std = std(logMdotBoundedData) # Get the standard deviation of the above set
            logMdot_spread_from_mean = abs.(logMdotBoundedData.-logMdotBoundedData_mean) # Calculate how far each mass-loss rate is from the mean value
            #print("percentile_to_trim*num_in_bound=",percentile_to_trim*num_in_bound,"\n")
            num_to_check = round(Int16,percentile_to_trim*num_in_bound) # Get the number of values we want to trim using percentile_trim, and convert this to an integer. If need be, we can change the memory limit to be larger, but 2^15 should be plenty. 
            ranked_logMdot_spread_from_mean = sortperm(logMdot_spread_from_mean) # What are the ranked indices of logMdot_spread_from_mean
            indices_to_trim = []
            #print("num_to_check=",num_to_check,"\n")
            for l in 1:num_to_check
                index_to_cut = indexin(l,ranked_logMdot_spread_from_mean)
                #print("index_to_cut=",index_to_cut,"\n")
                append!(indices_to_trim,index_to_cut)
            end
            indices_to_trim = sort(indices_to_trim)
            #print("indices_to_trim=",indices_to_trim)
            #indices_to_trim = sortperm(logMdot_spread_from_mean)[length(logMdot_spread_from_mean)-num_to_check,length(logMdot_spread_from_mean),num_to_check] # Determine the indices of the data values we're going to trim
            trimmed_logMdotBoundedData = deleteat!(logMdotBoundedData,indices_to_trim) # Cut the rows specified above
            #print("maximum(trimmed_logMdotBoundedData):",maximum(trimmed_logMdotBoundedData),"\n")
            #print("minimum(trimmed_logMdotBoundedData):",minimum(trimmed_logMdotBoundedData),"\n")
            delta_logMdot = maximum(trimmed_logMdotBoundedData) - minimum(trimmed_logMdotBoundedData) # With the extremes removed, calculate the width of logMdot
            #print("Δlog(Mdot)=",delta_logMdot,"\n")
            append!(deltas_array, [delta_logL, delta_logy, delta_logMdot])
            # Using PMR and RML relations to determine the exponents for the other power law, with the correct chemistry and 
            # pulsation mode
            if (chemistry == "M") | (chemistry == "oxygen-rich") # if the chemistry is oxygen-rich 
                ARLM = 10^(-0.241)
                BRLM = 0.690
                CRLM = -0.251
                if (pulsation_mode == "0") | (pulsation_mode == "fundamental")
                    APMR = 10^(-2.234)
                    BPMR = -0.7802
                    CPMR = 2.027
                elseif (pulsation_mode == "1") | (pulsation_mode == "first-overtone")
                    APMR = 10^(-1.554)
                    BPMR = -0.529
                    CPMR = 1.570
                end
            elseif (chemistry == "C") | (chemistry == "carbon-rich") # if the chemistry is carbon-rich
                ARLM = 10^(-0.323)
                BRLM = 0.737
                CRLM = -0.371
                if (pulsation_mode == "0") | (pulsation_mode == "fundamental")
                    APMR = 10^(-2.285)
                    BPMR = 0.826
                    CPMR = 2.043
                elseif (pulsation_mode == "1") | (pulsation_mode == "first-overtone")
                    APMR = 10^(-1.554)
                    BPMR = -0.528
                    CPMR = 1.570
                end
            end
            if y_data_type == "M" # Give the exponents B, C for Mdot = A L^B M^-C
                B = -delta_logMdot/delta_logL
                B_M = -delta_logMdot/delta_logL 
                #print("B_M = ",B_M)
                C = -delta_logMdot/delta_logy
                C_M = -delta_logMdot/delta_logy 
                #print("C_M = ",C_M)
                C_P = (C/(BPMR+CPMR))
                B_P = (B_M - C_P*(CPMR*BRLM))
                #print("B_P = ",B_P)
                #print("C_P = ",C_P)
            elseif y_data_type == "P" # Give the exponents B, C for Mdot = A L^B P^C
                B = -delta_logMdot/delta_logL
                B_P = -delta_logMdot/delta_logL
                #print("B_P = ",B_P)
                C = -delta_logMdot/delta_logy
                C_P = -delta_logMdot/delta_logy
                #print("C_P = ",C_P)
                B_M = B_P + C_P*CPMR*BRLM
                C_M = (C_P * BPMR + C_P*CPMR*CRLM)
                #print("B_M = ",B_M)
                #print("C_M = ",C_M)
            end
            #= To determine the coefficient of the power law, we will force A to be whatever value is necessary for the
            mean residual to be 0. We can freely transform this values as well between the types of power laws using
            RML and PMR relations. =#
            A = 0 # Initialize A
            samples = 0 # Initialize a counter
            for k in 1:ncol(star_data)
                if (isnan(logMdotData[k]) == true) | (isnan(logLData[k]) == true) | (isnan(logyData[k]) == true)
                    continue
                else
                    A = A + logMdotData[k] - B*logLData[k] - C*logyData[k] # Calculate an initial guess for A. The assignment of B and C above should handle the difference in versions above.
                end
            end
            A = A/samples
            logMdot_predict = A .+ B.*logLData + C.*logyData
            correction = mean(logMdotData-logMdot_predict)
            A = A + correction # Apply the correction to A
            calc_logMdot = A .+ B.*logLData .+ C.*logyData
            if y_data_type == "M"
                A_M = 10^A
                A_P = 10^(log10(A_M) - (C/(BPMR-CPMR)*log10(APMR)+CPMR*log10(ARLM)))
            elseif y_data_type == "P"
                A_P = 10^A
                A_M = 10^(log10(A_P) + C_P*log10(APMR) + C_P*CPMR*log10(ARLM))
            end
            Exponent_array = [[A_P, B_P, C_P], [A_M, B_M, C_M]]
        end
        append!(Exponents_array,Exponent_array)
        fit = lm(@formula(Y ~ X), DataFrame(X=logMdotData, Y=calc_logMdot))
        
        fit_params = coef(fit)
        fit_stderror = stderror(fit)
        append!(fit_qual_array,fit_params[2])
        append!(stderror_array,fit_stderror)
        n = n + delta_n
    end # End While loop 
    fit_qual_testing = abs.(fit_qual_array .- 1)
    where_min = argmin(fit_qual_testing)
    
    print("Optimal fit found at ",where_min,"\n")
    print("n=",n_array[where_min],"\n")
    print("Fit quality: Slope=",fit_qual_array[where_min],"\n")
    print("Fit standard dev: sigma=", stderror_array[where_min],"\n")
    print("delta logL, delta logM, delta logMdot:", deltas_array[where_min],"\n")
    print("[A_P, B_P, C_P], [A_M, B_M, C_M]:", Exponents_array[where_min],"\n")
    print("# Stars in Bound:", stars_in_bound_array[where_min],"\n")
    print("\n","\n")
    
    print("Lower bound fit:")
    print("n=",n_array[1],"\n")
    print("Fit quality: Slope=",fit_qual_array[1],"\n")
    print("Fit standard dev: sigma=", stderror_array[1],"\n")
    print("delta logL, delta logM, delta logMdot:", deltas_array[1],"\n")
    print("[A_P, B_P, C_P], [A_M, B_M, C_M]:", Exponents_array[1],"\n")
    print("# Stars in Bound:", stars_in_bound_array[1],"\n")
    print("\n","\n")
    
    print("Upper bound fit:")
    print("n=",last(n_array),"\n")
    print("Fit quality: Slope=",last(fit_qual_array),"\n")
    print("Fit standard dev: sigma=", last(stderror_array),"\n")
    print("delta logL, delta logM, delta logMdot:", last(deltas_array),"\n")
    print("[A_P, B_P, C_P], [A_M, B_M, C_M]:", last(Exponents_array),"\n")
    print("# Stars in Bound:", last(stars_in_bound_array),"\n")
    print("\n","\n")
    return Exponents_array[where_min]
end # End Function

fit_seek_one_percentile_m (generic function with 8 methods)

In [289]:
fit_seek_one_percentile_m(M0StarData , "P")

deathline_fit:StatsModels.TableRegressionModel{LinearModel{GLM.LmResp{Vector{Float64}}, GLM.DensePredChol{Float64, LinearAlgebra.CholeskyPivoted{Float64, Matrix{Float64}, Vector{Int64}}}}, Matrix{Float64}}

Y ~ 1 + X

Coefficients:
──────────────────────────────────────────────────────────────────────────
                  Coef.  Std. Error      t  Pr(>|t|)  Lower 95%  Upper 95%
──────────────────────────────────────────────────────────────────────────
(Intercept)  -0.0140119   0.0398121  -0.35    0.7249   -0.09209  0.0640662
X             0.62598     0.0112585  55.60    <1e-99    0.6039   0.64806
──────────────────────────────────────────────────────────────────────────
Optimal fit found at 1
n=0.5
Fit quality: Slope=NaN
Fit standard dev: sigma=NaN
delta logL, delta logM, delta logMdot:0.11458352162143814
[A_P, B_P, C_P], [A_M, B_M, C_M]:[NaN, -18.874679944319862, -30.152213476583164]
# Stars in Bound:651


Lower bound fit:n=0.5
Fit quality: Slope=NaN
Fit standard dev: sigma=NaN
delta

3-element Vector{Float64}:
 NaN
 -18.874679944319862
 -30.152213476583164

In [292]:
fit_seek_one_percentile_m(M0StarData , "P", "M", "0", false, 0.5, 3.0, 0.01, 0.05)

deathline_fit:StatsModels.TableRegressionModel{LinearModel{GLM.LmResp{Vector{Float64}}, GLM.DensePredChol{Float64, LinearAlgebra.CholeskyPivoted{Float64, Matrix{Float64}, Vector{Int64}}}}, Matrix{Float64}}

Y ~ 1 + X

Coefficients:
──────────────────────────────────────────────────────────────────────────
                  Coef.  Std. Error      t  Pr(>|t|)  Lower 95%  Upper 95%
──────────────────────────────────────────────────────────────────────────
(Intercept)  -0.0140119   0.0398121  -0.35    0.7249   -0.09209  0.0640662
X             0.62598     0.0112585  55.60    <1e-99    0.6039   0.64806
──────────────────────────────────────────────────────────────────────────
Optimal fit found at 1
n=0.5
Fit quality: Slope=NaN
Fit standard dev: sigma=NaN
delta logL, delta logM, delta logMdot:0.11458352162143814
[A_P, B_P, C_P], [A_M, B_M, C_M]:[NaN, -18.874679944319862, -30.152213476583164]
# Stars in Bound:651


Lower bound fit:n=0.5
Fit quality: Slope=NaN
Fit standard dev: sigma=NaN
delta

3-element Vector{Float64}:
 NaN
 -18.874679944319862
 -30.152213476583164

~~~
def fit_seek_one_percentile_m_logL_logP (star_data, initial_n=0.5, terminal_n=3.0, delta_n=0.1,percentile_to_trim = 0.05, 
              spec_mode_label ='', digit_precision = 6,text_x=0.03):
    '''
    Goal: script that calculates A, B, C for fit. Then, take those values and compare them to ideal. Knowing that,
    compare to previous fit. If better, continue. If worse, increase resolution then continue from previous.
    Repeat until a limit on the number of iterations is reached or we are as close to 1 as we can measure with
    numerical precision.
    '''
    '''
    First, we will handle all of the calculations that are simply properties of the data set. No need to recalculate
    these on every iteration. These calculations are finding the line of best fit for the data (presumably the deathline),
    finding the standard deviation of the data around the deathline, calculating the vertical displacement of the upper and 
    lower bounds given a multiplier n, calculating the horizontal displacement of the upper and lower bounds from the fit line,
    using those two calculations to determine the height and width of the strip (respectively), and sorting out the data found 
    inside our LM strip.
    '''
    logLData = np.log10(star_data[LIndex])
    logMData = np.log10(star_data[MIndex])
    logPData = np.log10(star_data[PIndex])
    logMdotData = np.log10(star_data[MdotIndex])     
    # Take data, fit logM v logL (this line is the deathline)
    deathline_fit = linear_fit_1var(logLData,logPData)
    deathline_coeffs = deathline_fit[0]
    print('deathline_fit:',deathline_fit)
    # Plot logM v logL and fit line for checking
    line_x_vals = np.asarray([np.nanmin(logLData),np.nanmax(logLData)])
    line_y_vals = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals
    #bound_size = std_multiplier*np.nanstd(np.log10(star_data[MIndex]))
    stnd_dev_around_DL = np.nanstd(logPData-deathline_coeffs[0]-deathline_coeffs[1]*logLData)
                                   
    i = 0
    n = initial_n
    #delta_n = 0.1 # An initial incrementing value. We'll start with rough adjustment and change this as we get closer to a good fit
    early_cancel_flag = False
    
    n_array = []
    Exponents_array = []
    fit_qual_array = []
    stand_dev_array = []
    rsq_array = []
    deltas_array = []
    stars_in_bound_array = []
    while n<=terminal_n:
        n_array.append(n)
        bound_size = n*stnd_dev_around_DL*np.cos(np.arctan(deathline_coeffs[1]))
        line_y_vals_up = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals + bound_size
        line_y_vals_down = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals - bound_size
        # Find width of deltaLogL and deltaLogM using fit line, and plot them
        logL_left = (np.average(line_y_vals) - deathline_coeffs[0]-bound_size)/deathline_coeffs[1]
        logL_right = (np.average(line_y_vals) - deathline_coeffs[0]+bound_size)/deathline_coeffs[1]
        delta_logL = logL_right - logL_left
        delta_logP = line_y_vals_up[1] - line_y_vals_down[1]
        star_data_inside_bound = np.asarray([])
        for l in range(0, len(star_data[MIndex])):
            if logPData[l] <= deathline_coeffs[0] + deathline_coeffs[1]*logLData[l] + bound_size:
                if logPData[l] >= deathline_coeffs[0] + deathline_coeffs[1]*logLData[l] - bound_size:
                    if len(star_data_inside_bound) == 0:
                        star_data_inside_bound = np.transpose(star_data)[l]
                    else:
                        star_data_inside_bound = np.vstack((star_data_inside_bound,np.transpose(star_data)[l]))
        star_data_inside_bound = np.transpose(star_data_inside_bound)
        logMdotBoundedData = np.log10(star_data_inside_bound[MdotIndex])
        logMdotBoundedData_mean = np.nanmean(logMdotBoundedData)
        logMdotBoundedData_std = np.nanstd(logMdotBoundedData)
        logMBoundedData = np.log10(star_data_inside_bound[MIndex])
        logPBoundedData = np.log10(star_data_inside_bound[PIndex])
        logLBoundedData = np.log10(star_data_inside_bound[LIndex])
        num_in_bound = len(logMdotBoundedData)
        stars_in_bound_array.append(num_in_bound)

        '''
        Now onto the loop. For some number of iterations, we are going to seek better fits for logMdot = A + BlogL + ClogM.
        From previous experimentation (see above function, fit_golf), we know that as we adjust m/n, the significance of the fit
        is more-or-less fixed using this method. Therefore, we can keep iterating to improve the quality of the fit. We are doing
        this over n and are adjusting m to be to trim the most extreme mass-loss rates
        '''
        logMdot_spread_from_mean = np.abs(logMdotBoundedData - logMdotBoundedData_mean)
        num_to_check = int(np.round_(percentile_to_trim*len(logMdot_spread_from_mean)))
        #print('num_to_check:',num_to_check)
        indices_to_trim = logMdot_spread_from_mean.argsort()[-num_to_check:]
        trimmed_logMdotBoundedData = np.delete(logMdotBoundedData,indices_to_trim)
        delta_logMdot = np.nanmax(trimmed_logMdotBoundedData)-np.nanmin(trimmed_logMdotBoundedData)
        deltas_array.append([delta_logL, delta_logP,delta_logMdot])
        #delta_logMdot = m*logMdotBoundedData_std
        # Calculate B = deltaLogMdot/deltaLogL and C = deltaLogMdot/deltaLogM
        B = -delta_logMdot/delta_logL
        C = delta_logMdot/delta_logP
        # Simpler, we can force A to be whatever is necessary for the mean residual to be 0
        # <logMdot_obsv - logMdot_calc> = 0 = sum(logMdot_obsv[i] - (A + BlogL[i] + ClogM[i]))/len(data)  
        # => <A> = sum(logMdot_obsv[i] - BlogL[i] - ClogM[i])/len(data)
        A = 0
        samples = 0
        for k in range(0, np.shape(star_data)[1]):
            if np.isnan(logMdotData[k])==True:
                #print('Index ',i,' skipped due to NaN in logMdot')
                continue
            elif np.isnan(logLData[k])==True:
                #print('Index ',i,' skipped due to NaN in logL')
                continue
            elif np.isnan(logPData[k])==True:
                #print('Index ',i,' skipped due to NaN in logM')
                continue
            else:
                A = A + logMdotData[k]-B*logLData[k] - C*logPData[k]
                samples = samples+1
        A = A/samples
        logMdot_predict = A + B*logLData + C*logPData
        correction = np.nanmean(logMdotData-logMdot_predict)
        A = A + correction
        calc_logMdot = A + B*logLData + C*logPData
        Exponents_array.append([A,B,C])
        fit = linear_fit_1var(logMdotData, calc_logMdot)
        #print('fit:',fit)
        fit_params = fit[0]
        fitRSq = fit[1]
        fit_qual_array.append(fit_params[1])
        rsq_array.append(fitRSq)
        residuals = logMdotData-calc_logMdot
        stand_dev_array.append( np.nanstd(residuals))
    
        n = n + delta_n
    fit_qual_array = np.asarray(fit_qual_array)
    fit_qual_testing = np.abs(fit_qual_array - 1)
    where_min = np.argmin(fit_qual_testing)

    
    active_exponents_array = Exponents_array[where_min]
    if spec_mode_label == 'M0':
        logAPMR = -2.234
        BPMR = -0.7802
        CPMR = 2.027
        logARLM = -0.241
        BRLM = 0.690
        CRLM = -0.251
        logALP = active_exponents_array[0]
        BLP = active_exponents_array[1]
        CLP = active_exponents_array[2]
    
        logA_LM = logALP + CLP*logAPMR + CLP*CPMR*logARLM
        B_LM = BLP+BRLM*CLP*CPMR
        C_LM = BPMR*CLP+CLP*CPMR*CRLM
        LM_exponents_list = [logA_LM, B_LM, C_LM]
    if spec_mode_label == 'M1':
        logAPMR = -1.554
        BPMR = -0.529
        CPMR = 1.570
        logARLM = -0.241
        BRLM = 0.690
        CRLM = -0.251
        logALP = active_exponents_array[0]
        BLP = active_exponents_array[1]
        CLP = active_exponents_array[2]
    
        logA_LM = logALP + CLP*logAPMR + CLP*CPMR*logARLM
        B_LM = BLP+BRLM*CLP*CPMR
        C_LM = BPMR*CLP+CLP*CPMR*CRLM
        LM_exponents_list = [logA_LM, B_LM, C_LM]
        
    if spec_mode_label == 'C0 and C0x':
        logAPMR = -2.285
        BPMR = -0.826
        CPMR = 2.043
        logARLM = -0.323
        BRLM = 0.737
        CRLM = -0.371
        logALP = active_exponents_array[0]
        BLP = active_exponents_array[1]
        CLP = active_exponents_array[2]
    
        logA_LM = logALP + CLP*logAPMR + CLP*CPMR*logARLM
        B_LM = BLP+BRLM*CLP*CPMR
        C_LM = BPMR*CLP+CLP*CPMR*CRLM
        LM_exponents_list = [logA_LM, B_LM, C_LM]
        
    if spec_mode_label == 'C1':
        logAPMR = -1.554
        BPMR = -0.529
        CPMR = 1.570
        logARLM = -0.323
        BRLM = 0.737
        CRLM = -0.371
        logALP = active_exponents_array[0]
        BLP = active_exponents_array[1]
        CLP = active_exponents_array[2]
    
        logA_LM = logALP + CLP*logAPMR + CLP*CPMR*logARLM
        B_LM = BLP+BRLM*CLP*CPMR
        C_LM = BPMR*CLP+CLP*CPMR*CRLM
        LM_exponents_list = [logA_LM, B_LM, C_LM]
    
    
    
    print('Optimal fit found at ',where_min)
    print('n=',n_array[where_min])
    print('Fit quality: Slope=',fit_qual_array[where_min])
    print('Fit significance: R^2=', rsq_array[where_min])
    print('Fit standard dev: sigma=', stand_dev_array[where_min])
    print('delta logL, delta logM, delta logMdot:', deltas_array[where_min])
    print('log A, B, C:', Exponents_array[where_min])
    print('log A, B, C (LM Strip):',LM_exponents_list)
    print('# Stars in Bound:', stars_in_bound_array[where_min])
    print('\n')
    
    print('Lower bound fit:')
    print('n=',n_array[0])
    print('Fit quality: Slope=',fit_qual_array[0])
    print('Fit significance: R^2=', rsq_array[0])
    print('Fit standard dev: sigma=', stand_dev_array[0])
    print('delta logL, delta logM, delta logMdot:', deltas_array[0])
    print('log A, B, C:', Exponents_array[0])
    print('# Stars in Bound:', stars_in_bound_array[0])
    print('\n')
    
    print('Upper bound fit:')
    print('n=',n_array[-1])
    print('Fit quality: Slope=',fit_qual_array[-1])
    print('Fit significance: R^2=', rsq_array[-1])
    print('Fit standard dev: sigma=', stand_dev_array[-1])
    print('delta logL, delta logM, delta logMdot:', deltas_array[-1])
    print('log A, B, C:', Exponents_array[-1])
    print('# Stars in Bound:', stars_in_bound_array[-1])
    print('\n')
    
    # Get all of the looping values with our optimal n
    n= n_array[where_min]
    bound_size = n*stnd_dev_around_DL*np.cos(np.arctan(deathline_coeffs[1]))
    line_y_vals_up = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals + bound_size
    line_y_vals_down = deathline_coeffs[0] + deathline_coeffs[1]*line_x_vals - bound_size
    # Find width of deltaLogL and deltaLogM using fit line, and plot them
    logL_left = (np.average(line_y_vals) - deathline_coeffs[0]-bound_size)/deathline_coeffs[1]
    logL_right = (np.average(line_y_vals) - deathline_coeffs[0]+bound_size)/deathline_coeffs[1]
    delta_logL = logL_right - logL_left
    delta_logP = line_y_vals_up[1] - line_y_vals_down[1]
    star_data_inside_bound = np.asarray([])
    for l in range(0, len(star_data[PIndex])):
        if logPData[l] <= deathline_coeffs[0] + deathline_coeffs[1]*logLData[l] + bound_size:
            if logPData[l] >= deathline_coeffs[0] + deathline_coeffs[1]*logLData[l] - bound_size:
                if len(star_data_inside_bound) == 0:
                    star_data_inside_bound = np.transpose(star_data)[l]
                else:
                    star_data_inside_bound = np.vstack((star_data_inside_bound,np.transpose(star_data)[l]))
    star_data_inside_bound = np.transpose(star_data_inside_bound)
    #print('star_data_inside_bound:',star_data_inside_bound)
    logMdotBoundedData = np.log10(star_data_inside_bound[MdotIndex])
    logMdotBoundedData_mean = np.nanmean(logMdotBoundedData)
    logMdotBoundedData_std = np.nanstd(logMdotBoundedData)
    logMBoundedData = np.log10(star_data_inside_bound[MIndex])
    logPBoundedData = np.log10(star_data_inside_bound[PIndex])
    logLBoundedData = np.log10(star_data_inside_bound[LIndex])
    logMdot_spread_from_mean = np.abs(logMdotBoundedData - logMdotBoundedData_mean)
    num_to_check = int(np.round_(percentile_to_trim*len(logMdot_spread_from_mean)))
    #print('num_to_check:',num_to_check)
    indices_to_trim = logMdot_spread_from_mean.argsort()[-num_to_check:]
    #print('indices_to_trim',indices_to_trim)
    trimmed_logMdotBoundedData = np.delete(logMdotBoundedData,indices_to_trim)
    #print('Indices removed?:')
    delta_logMdot = np.nanmax(trimmed_logMdotBoundedData)-np.nanmin(trimmed_logMdotBoundedData)
    deltas_array.append([delta_logL, delta_logP,delta_logMdot])
    #delta_logMdot = m*logMdotBoundedData_std
    # Calculate B = deltaLogMdot/deltaLogL and C = deltaLogMdot/deltaLogM
    B = -delta_logMdot/delta_logL
    C = delta_logMdot/delta_logP
    # Simpler, we can force A to be whatever is necessary for the mean residual to be 0
    # <logMdot_obsv - logMdot_calc> = 0 = sum(logMdot_obsv[i] - (A + BlogL[i] + ClogM[i]))/len(data)  
    # => <A> = sum(logMdot_obsv[i] - BlogL[i] - ClogM[i])/len(data)
    A = 0
    for k in range(0, np.shape(star_data)[1]):
        if np.isnan(logMdotData[k])==True:
            #print('Index ',i,' skipped due to NaN in logMdot')
            continue
        elif np.isnan(logLData[k])==True:
            #print('Index ',i,' skipped due to NaN in logL')
            continue
        elif np.isnan(logPData[k])==True:
            #print('Index ',i,' skipped due to NaN in logM')
            continue
        else:
            A = A + logMdotData[k]-B*logLData[k] - C*logPData[k]
    A = A/np.shape(star_data)[1]
    logMdot_predict = A + B*logLData + C*logPData
    correction = np.nanmean(logMdotData-logMdot_predict)
    A = A + correction
    calc_logMdot = A + B*logLData + C*logPData
    fit = linear_fit_1var(logMdotData, calc_logMdot)
    fit_params = fit[0]
    fitRSq = fit[1]
    residuals = logMdotData-calc_logMdot
    
    
    # Plot 1: The LM Strip
    fig, ax = plt.subplots()
    plot.scatter(logLData, logPData,s=2)
    plot.plot(line_x_vals,line_y_vals,linestyle='-',color='black')
    plot.plot(line_x_vals,line_y_vals_up,linestyle='--',color='black')
    plot.plot(line_x_vals,line_y_vals_down,linestyle='--',color='black')
    plot.xlabel(r'$\log L$')
    plot.ylabel(r'$\log P$')
    #plot.title(spec_mode_label)
    ax.text(text_x, .97,spec_mode_label,horizontalalignment='center',verticalalignment='center',transform = ax.transAxes,fontsize=10)
    plot.vlines(np.average(line_x_vals),np.average(line_y_vals)-delta_logP/2,np.average(line_y_vals)+delta_logP/2,color='tab:green',linestyle='-',linewidth=4)
    plot.plot([logL_left,logL_right],[np.average(line_y_vals),np.average(line_y_vals)],color='tab:purple',linewidth=4)
    if type(n) == float:
        plot_name = 'deathzone_'+spec_mode_label+'_stnd_dev_'+str(n)[:4]
    #elif std_multiplier/(1/2*std_multiplier) == 2:
    #    plot_name = 'deathzone_'+spec_mode_label+'_stnd_dev_of_all_double'
    else:
        plot_name = 'deathzone_'+spec_mode_label+'_stnd_dev_of_all'
    plot.savefig(plot_name+'.png')
    plot.savefig(plot_name+'.pdf') # Why do journals require this format...
    plot.show()
    plot.close('all')
    
    # Plot 2: logMdot Histograms
    fig, ax = plt.subplots()
    n_hist, bins_hist, patches_hist = plot.hist(np.log10(star_data_inside_bound[MdotIndex]))
    ax.text(text_x, .97,spec_mode_label,horizontalalignment='center',verticalalignment='center',transform = ax.transAxes,fontsize=10)
    print('Full range:',np.nanmax(logMdotBoundedData),np.nanmin(logMdotBoundedData))
    print('vlines locations:',np.nanmax(trimmed_logMdotBoundedData),np.nanmin(trimmed_logMdotBoundedData))
    plot.vlines([np.nanmax(trimmed_logMdotBoundedData),np.nanmin(trimmed_logMdotBoundedData)],0,np.nanmax(n_hist))
    plot.xlabel(r'$\log \dot{M}$')
    if type(n) == int:
        plot_name = 'logMdot_histogram_'+spec_mode_label+'_stnd_dev_'+str(n)
    elif type(n) == float:
        plot_name = 'logMdot_histogram_'+spec_mode_label+'_stnd_dev_'+str(n)[0:5]
    else:
        plot_name = 'logMdot_histogram_'+spec_mode_label+'_stnd_dev_of_all'
    #plot.legend(loc=4)
    plot.savefig(plot_name+'.png')
    plot.savefig(plot_name+'.pdf') # Why do journals require this format...
    
    # Plot 3: log(Mdot/M) Histograms
    fig, ax = plt.subplots()
    n_hist, bins_hist, patches_hist = plot.hist(np.log10(star_data_inside_bound[MdotIndex]/star_data_inside_bound[MIndex]))
    ax.text(text_x, .97,spec_mode_label,horizontalalignment='center',verticalalignment='center',transform = ax.transAxes,fontsize=10)
    #print('Full range:',np.nanmax(logMdotBoundedData),np.nanmin(logMdotBoundedData))
    #print('vlines locations:',np.nanmax(trimmed_logMdotBoundedData),np.nanmin(trimmed_logMdotBoundedData))
    #plot.vlines([np.nanmax(trimmed_logMdotBoundedData),np.nanmin(trimmed_logMdotBoundedData)],0,np.nanmax(n_hist))
    plot.xlabel(r'$\log (\dot{M}/M)$')
    if type(n) == int:
        plot_name = 'logMdot_per_M_histogram_'+spec_mode_label+'_stnd_dev_'+str(n)
    elif type(n) == float:
        plot_name = 'logMdot_per_M_histogram_'+spec_mode_label+'_stnd_dev_'+str(n)[0:5]
    else:
        plot_name = 'logMdot_per_M_histogram_'+spec_mode_label+'_stnd_dev_of_all'
    #plot.legend(loc=4)
    plot.savefig(plot_name+'.png')
    plot.savefig(plot_name+'.pdf') # Why do journals require this format...
    
    
    return A, B, C
~~~